# MedSAM (zero-shot then fine-tuned) - BTXRD

Single file, two parts, run top to bottom:
- **Part A**: official pretrained `medsam_vit_b` checkpoint, box prompt,
  no fine-tuning - zero-shot reference number.
- **Part B**: fine-tunes only the mask decoder (image encoder and prompt
  encoder frozen), following the official MedSAM paper's own recipe,
  model-selects on the validation split under the off-center condition
  (matching every other model in this article), then re-tests the best
  checkpoint the same way as Part A.

Both parts use the exact same box protocol as every other model in this
article (`_center_zoom_bbox` = covering, `_center_shift_bbox` = off-center),
reused directly from `PromptSegmentationDataset` in the PGA-UNet codebase, and
report the same six metrics (Dice, IoU, Precision, Recall, HD95, CBL).

In [ ]:
# -- Setup -----------------------------------------------------------------
%cd /kaggle/working
import os, gdown, torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# PGA-UNet repo: only needed for its dataset.py (box-protocol + polygon listing),
# not for any PGA-UNet model code.
if not os.path.exists('PGA_Unet2D'):
    !git clone --branch main --single-branch https://github.com/ThongLuc2k3/PGA_Unet2D.git
PGA_PATH = '/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'

# MedSAM repo (official bowang-lab implementation)
if not os.path.exists('MedSAM'):
    !git clone --branch main --single-branch https://github.com/bowang-lab/MedSAM.git
!pip install -q -e MedSAM

DATASET_ID   = '1Ep3w8GujpCQ5Qb6nlXKxXhsCxlc3k8gx'
DATASET_ROOT = 'dataset_BTXRD'
if not os.path.exists(f'{PGA_PATH}/{DATASET_ROOT}'):
    gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}',
                   f'/kaggle/working/{DATASET_ROOT}.zip', quiet=False)
    !unzip -oq /kaggle/working/{DATASET_ROOT}.zip -d {PGA_PATH}/

# MedSAM checkpoint (official pretrained weights, box-prompt only, no fine-tuning)
MEDSAM_DIR = '/kaggle/working/medsam_ckpt'
os.makedirs(MEDSAM_DIR, exist_ok=True)
MEDSAM_CKPT_PATH = f'{MEDSAM_DIR}/medsam_vit_b.pth'
if not os.path.exists(MEDSAM_CKPT_PATH):
    # Folder id from the shared Drive link - downloads the whole folder, then we
    # locate medsam_vit_b.pth inside it.
    MEDSAM_CKPT_FOLDER_ID = '1ETWmi4AiniJeWOt6HAsYgTjYv_fkgzoN'
    gdown.download_folder(id=MEDSAM_CKPT_FOLDER_ID, output=MEDSAM_DIR, quiet=False, use_cookies=False)
    found = [os.path.join(r, f) for r, _, fs in os.walk(MEDSAM_DIR) for f in fs if f == 'medsam_vit_b.pth']
    assert found, f'medsam_vit_b.pth not found under {MEDSAM_DIR} after download'
    if found[0] != MEDSAM_CKPT_PATH:
        os.rename(found[0], MEDSAM_CKPT_PATH)
assert os.path.exists(MEDSAM_CKPT_PATH)
TRAIN_IMG  = f'/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/{DATASET_ROOT}/train/images'
TRAIN_JSON = f'/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/{DATASET_ROOT}/train/annotations'
VAL_IMG    = f'/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/{DATASET_ROOT}/val/images'
VAL_JSON   = f'/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/{DATASET_ROOT}/val/annotations'
print(f'\nSetup complete | dataset=BTXRD | checkpoint {os.path.getsize(MEDSAM_CKPT_PATH)//1024//1024} MB')

In [ ]:
# -- Model + shared helpers -------------------------------------------------
import sys, csv, json as _json
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from scipy.ndimage import binary_erosion, distance_transform_edt

if PGA_PATH not in sys.path:
    sys.path.insert(0, PGA_PATH)
from dataset import PromptSegmentationDataset

sys.path.insert(0, '/kaggle/working/MedSAM')
from segment_anything import sam_model_registry

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DS_NAME = 'BTXRD'
TEST_IMG  = f'{PGA_PATH}/{DATASET_ROOT}/test/images'
TEST_JSON = f'{PGA_PATH}/{DATASET_ROOT}/test/annotations'
RESULT_DIR = f'{PGA_PATH}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

medsam_model = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT_PATH).to(DEVICE)
medsam_model.eval()

# Reused only for its box-protocol methods (_center_zoom_bbox/_center_shift_bbox)
# and its polygon listing (all_samples), NOT for its tensor preprocessing - MedSAM
# needs its own 1024x1024 preprocessing, done separately below.
box_ds = PromptSegmentationDataset(TEST_IMG, TEST_JSON, is_train=False, prompt_mode='center_zoom')
print(f'{DS_NAME}: {len(box_ds.all_samples)} (image, polygon) test samples')

@torch.no_grad()
def medsam_embed(img_3c):
    # img_3c: HxWx3 uint8. Returns (embedding, H, W) at original resolution.
    H, W, _ = img_3c.shape
    img_1024 = cv2.resize(img_3c, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), 1e-8, None)
    img_1024_t = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    embedding = medsam_model.image_encoder(img_1024_t)
    return embedding, H, W

@torch.no_grad()
def medsam_predict_prob(img_embed, box_xyxy, H, W):
    # box_xyxy: [x0, y0, x1, y1] in ORIGINAL image pixel coordinates.
    # Returns a float32 probability map at the ORIGINAL (H, W) resolution.
    box_1024 = np.array(box_xyxy, dtype=np.float32) / np.array([W, H, W, H]) * 1024.0
    box_t = torch.as_tensor(box_1024, dtype=torch.float32, device=DEVICE)[None, None, :]
    sparse_emb, dense_emb = medsam_model.prompt_encoder(points=None, boxes=box_t, masks=None)
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_emb,
        dense_prompt_embeddings=dense_emb,
        multimask_output=False,
    )
    prob = torch.sigmoid(low_res_logits)
    prob = F.interpolate(prob, size=(H, W), mode='bilinear', align_corners=False)
    return prob.squeeze().float().cpu().numpy()

# -- Metrics: identical formulas to the rest of the article --
def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any():
        return 0.0
    S = max(pred.shape)
    if not p.any() or not g.any():
        return float(S)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(S) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img(pred_bin, gt_bin, eps=1e-6):
    pm, gm = pred_bin.astype(np.float32), gt_bin.astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float(tp/(tp+fp+eps)), recall=float(tp/(tp+fn+eps)), hd95=hd95, cbl=cbl)

print('helpers ready | device', DEVICE)
def run_test(tag):
    # Shared by Part A (zero-shot) and Part B (post-fine-tune): uses
    # medsam_model's LIVE weights, so this automatically reflects whatever
    # fine-tuning Part B has done by the time it is called a second time.
    from collections import defaultdict
    by_image = defaultdict(lambda: None)
    names = sorted(set(n for n, _ in box_ds.all_samples))
    for n_done, img_name in enumerate(names):
        base = os.path.splitext(img_name)[0]
        img_gray = cv2.imread(os.path.join(TEST_IMG, img_name), cv2.IMREAD_GRAYSCALE)
        H, W = img_gray.shape
        img_3c = np.repeat(img_gray[:, :, None], 3, axis=-1)
        embedding, _, _ = medsam_embed(img_3c)
        with open(os.path.join(TEST_JSON, base + '.json'), encoding='utf-8') as f:
            data = _json.load(f)
        pred_zoom  = np.zeros((H, W), dtype=np.float32)
        pred_shift = np.zeros((H, W), dtype=np.float32)
        gt_union   = np.zeros((H, W), dtype=np.uint8)
        poly_idxs = [i for i, s in enumerate(data.get('shapes', [])) if s.get('shape_type') == 'polygon']
        for shape_idx in poly_idxs:
            points = np.array(data['shapes'][shape_idx]['points'])
            cv2.fillPoly(gt_union, [points.astype(np.int32)], 1)
            x_min, y_min = points.min(axis=0)
            x_max, y_max = points.max(axis=0)
            sample_idx = box_ds.all_samples.index((img_name, shape_idx))
            bz = box_ds._center_zoom_bbox(x_min, x_max, y_min, y_max, H, W)
            bs = box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W, seed_idx=sample_idx)
            box_zoom  = [bz[0], bz[2], bz[1], bz[3]]
            box_shift = [bs[0], bs[2], bs[1], bs[3]]
            prob_zoom  = medsam_predict_prob(embedding, box_zoom, H, W)
            prob_shift = medsam_predict_prob(embedding, box_shift, H, W)
            pred_zoom  = np.maximum(pred_zoom, prob_zoom)
            pred_shift = np.maximum(pred_shift, prob_shift)
        by_image[img_name] = dict(pred_zoom=pred_zoom, pred_shift=pred_shift, gt=gt_union)
        if (n_done + 1) % 25 == 0:
            print(f'  {n_done + 1}/{len(names)} images')
    rows = []
    for mode, key in [('covering', 'pred_zoom'), ('off-center', 'pred_shift')]:
        per_image = []
        for img_name, rec in by_image.items():
            pred_bin = (rec[key] > 0.5).astype(np.uint8)
            m = calc_metrics_img(pred_bin, rec['gt'])
            m['image'] = img_name
            per_image.append(m)
        agg = {k: float(np.mean([r[k] for r in per_image])) for k in ('dice', 'iou', 'precision', 'recall', 'hd95', 'cbl')}
        rows.append(dict(dataset=DS_NAME, model=f'MedSAM ({tag})', prompt=mode, **agg))
        with open(f'{RESULT_DIR}/medsam_{tag.replace(chr(45),chr(95))}_{DS_NAME.lower()}_{mode.replace(chr(45),chr(95))}_per_image.csv',
                  'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(per_image[0].keys())); w.writeheader(); w.writerows(per_image)
    print(f'\n{DS_NAME}: MedSAM {tag}, image-level merged\n')
    print(f'{"prompt":<12}{"Dice":>8}{"IoU":>8}{"Prec":>8}{"Recall":>8}{"HD95":>8}{"CBL":>8}')
    print('-' * 60)
    for r in rows:
        print(f'{r["prompt"]:<12}{r["dice"]:>8.3f}{r["iou"]:>8.3f}{r["precision"]:>8.3f}'
              f'{r["recall"]:>8.3f}{r["hd95"]:>8.1f}{r["cbl"]:>8.3f}')
    with open(f'{RESULT_DIR}/medsam_{tag.replace(chr(45),chr(95))}_{DS_NAME.lower()}_summary.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    return rows, by_image

# -- Qualitative visualization: same 5-column style as
# Finetune_SAMMed2D_test_robust.ipynb (Input / Prompt / Prediction / GT / TP-FP-FN),
# 10 shared stems, one PNG + one export_qualitative_rows call per stem.
def visualize_qualitative(by_image_vis, img_dir, box_ds_vis, model_label, prefix_tag):
    import sys as _sys
    from pathlib import Path as _Path
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    if str(_Path(PGA_PATH)) not in _sys.path:
        _sys.path.insert(0, str(_Path(PGA_PATH)))
    from qualitative_visualization import export_qualitative_rows, select_shared_stems

    modes = [('covering', 'pred_zoom', 'limegreen'), ('off-center', 'pred_shift', 'tomato')]
    selection_records = [dict(img_name=n) for n in by_image_vis.keys()]
    vis_images = select_shared_stems(selection_records, n_multi=5, n_single=5)

    for vis_name in vis_images:
        if vis_name not in by_image_vis:
            continue
        rec = by_image_vis[vis_name]
        img_gray = cv2.imread(os.path.join(img_dir, vis_name), cv2.IMREAD_GRAYSCALE)
        H, W = img_gray.shape
        rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
        gt = rec['gt']
        ys, xs = np.where(gt > 0)
        gt_ov = rgb.copy()
        gt_ov[gt > 0] = np.clip(rgb[gt > 0] * 0.4 + np.array([0, 200, 0]) * 0.6, 0, 255)

        fig, axes = plt.subplots(len(modes), 5, figsize=(20, 4 * len(modes)))
        fig.suptitle(f'{model_label}: {vis_name}', fontsize=13, fontweight='bold', y=1.01)
        mode_records = []
        for row, (mode, key, color) in enumerate(modes):
            pred = (rec[key] > 0.5).astype(np.uint8)   # rec[key] is a float prob map; threshold before overlays
            pr_ov = rgb.copy()
            pr_ov[pred > 0] = np.clip(rgb[pred > 0] * 0.4 + np.array([220, 60, 60]) * 0.6, 0, 255)
            diff = rgb.copy()
            diff[gt > 0] = [0, 200, 0]
            diff[pred > 0] = [200, 60, 60]
            diff[(gt > 0) & (pred > 0)] = [220, 200, 0]

            axes[row, 0].imshow(img_gray, cmap='gray')
            axes[row, 0].set_ylabel(mode, fontsize=11, fontweight='bold', color=color,
                                    rotation=0, labelpad=65, va='center')
            axes[row, 1].imshow(img_gray, cmap='gray')
            if len(xs) > 0:
                x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
                if mode == 'covering':
                    bx0, bx1, by0, by1 = box_ds_vis._center_zoom_bbox(x0, x1, y0, y1, H, W)
                else:
                    bx0, bx1, by0, by1 = box_ds_vis._center_shift_bbox(x0, x1, y0, y1, H, W, seed_idx=row)
                axes[row, 1].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0,
                                                  linewidth=2.5, edgecolor=color, facecolor='none'))
            axes[row, 2].imshow(pr_ov)
            axes[row, 3].imshow(gt_ov)
            axes[row, 4].imshow(diff)
            mode_records.append(dict(img_name=f'{vis_name}_{mode}', gt=gt.copy(), pred=pred.copy()))
            for ax in axes[row]:
                ax.axis('off')

        plt.tight_layout()
        out = f'{prefix_tag}_{os.path.splitext(vis_name)[0]}.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        export_qualitative_rows(fig, axes, mode_records, prefix=f'{prefix_tag}_{vis_name}')
        print(f'saved {out}')


## Part A - zero-shot test (no fine-tuning)

Official pretrained weights as downloaded, box prompt only, both prompt
conditions, image-level merged.

In [ ]:
# -- Part A: run + report --
rows_zeroshot, images_zeroshot = run_test('zero-shot')

# -- Qualitative figures (10 shared stems, same style as Finetune_SAMMed2D_test_robust.ipynb) --
visualize_qualitative(images_zeroshot, TEST_IMG, box_ds, 'MedSAM (zero-shot)',
                      f'medsam_zeroshot_{DS_NAME.lower()}')

## Part B - fine-tune (mask decoder only, image encoder + prompt encoder frozen)

The official MedSAM recipe: freeze the heavy ViT image encoder and the
lightweight box prompt encoder, fine-tune only the mask decoder. Boxes
follow the same protocol as PGA-UNet and the other fine-tuned baselines
(`center_mixed`, 80% `center_shift` / 20% `center_zoom` per sample); each
baseline otherwise keeps its own official recipe, and no flip or rotation
augmentation is added here. Loss is computed at the decoder's native
256x256 resolution against a downsampled ground truth (the standard MedSAM
convention; full-resolution scoring is used only for evaluation), one
optimizer step per 4-polygon batch. Model selection is on the validation
split under the fixed off-center condition, early stopping with patience
30 (the fine-tuned SAM-Med2D baseline's longer window, not PGA-UNet's
from-scratch 15), up to 150 epochs, `lr=1e-5`.

In [ ]:
# -- Part B: fine-tuning setup --
import time
import torch.nn.functional as F
import random

EPOCHS     = 150
PATIENCE   = 30     # matches the fine-tuned SAM-Med2D baseline in this article, not PGA's own from-scratch 15
BATCH_SIZE = 4      # matches PGA-UNet's and SAM-Med2D's batch size (loss averaged over 4 samples per step)
LR         = 1e-5
MIXED_SHIFT_PROB = 0.8   # matches PromptSegmentationDataset's own default
LOG_EVERY  = 20          # optimizer steps between in-epoch progress prints
BEST_CKPT_PATH = f'{PGA_PATH}/checkpoints/medsam_finetuned_{DS_NAME.lower()}_best.pth'
os.makedirs(os.path.dirname(BEST_CKPT_PATH), exist_ok=True)

random.seed(22120196); np.random.seed(22120196); torch.manual_seed(22120196)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(22120196)

train_box_ds = PromptSegmentationDataset(TRAIN_IMG, TRAIN_JSON, is_train=True, prompt_mode='center_mixed')
val_box_ds   = PromptSegmentationDataset(VAL_IMG, VAL_JSON, is_train=False, prompt_mode='center_shift')
print(f'{DS_NAME}: {len(train_box_ds.all_samples)} train polygons, {len(val_box_ds.all_samples)} val polygons')

for p in medsam_model.image_encoder.parameters(): p.requires_grad = False
for p in medsam_model.prompt_encoder.parameters(): p.requires_grad = False
for p in medsam_model.mask_decoder.parameters(): p.requires_grad = True
medsam_model.image_encoder.eval()
medsam_model.prompt_encoder.eval()
medsam_model.mask_decoder.train(True)
optimizer = torch.optim.AdamW(medsam_model.mask_decoder.parameters(), lr=LR, weight_decay=1e-4)

# The image encoder and prompt encoder are frozen, so an image's 1024x1024
# embedding is identical on every epoch. Re-running the ViT-B encoder once per
# polygon per epoch (~2000x) is what makes an epoch take many silent minutes;
# cache each embedding on first use (fp16 on CPU, ~2 MB each) and reuse it, the
# way the official MedSAM fine-tuning script does. Epoch 1 still pays the encoder
# cost while the cache fills; later epochs are decoder-only. Switch the cache
# dtype to torch.float32 if you would rather trade ~2x host RAM for bit-exactness.
_embed_cache = {}
_json_cache = {}

_encode_t0 = None

@torch.no_grad()
def get_embedding(img_name, img_dir):
    global _encode_t0
    hit = _embed_cache.get(img_name)
    if hit is None:
        if _encode_t0 is None:
            _encode_t0 = time.time()
        g = cv2.imread(os.path.join(img_dir, img_name), cv2.IMREAD_GRAYSCALE)
        H, W = g.shape
        emb, _, _ = medsam_embed(np.repeat(g[:, :, None], 3, axis=-1))
        hit = (emb.detach().to('cpu', dtype=torch.float16), H, W)
        _embed_cache[img_name] = hit
        if len(_embed_cache) % 20 == 0:
            rate = len(_embed_cache) / max(time.time() - _encode_t0, 1e-6)
            print(f'    encoding images into cache: {len(_embed_cache)} done '
                  f'({rate:.1f}/s)', flush=True)
    emb, H, W = hit
    return emb.to(DEVICE, dtype=torch.float32), H, W

def get_json(img_name, json_dir):
    if img_name not in _json_cache:
        with open(os.path.join(json_dir, os.path.splitext(img_name)[0] + '.json'), encoding='utf-8') as f:
            _json_cache[img_name] = _json.load(f)
    return _json_cache[img_name]

def medsam_compute_loss(embedding, box_xyxy, gt_mask_hw, H, W):
    # embedding: cached [1, 256, 64, 64] float32 on DEVICE (image + prompt encoder
    # are frozen). Returns the loss TENSOR (no backward/step here) so the training
    # loop can average it with 3 other samples first, matching batch_size=4.
    with torch.no_grad():
        box_1024 = np.array(box_xyxy, dtype=np.float32) / np.array([W, H, W, H]) * 1024.0
        box_t = torch.as_tensor(box_1024, dtype=torch.float32, device=DEVICE)[None, None, :]
        sparse_emb, dense_emb = medsam_model.prompt_encoder(points=None, boxes=box_t, masks=None)
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=embedding,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_emb,
        dense_prompt_embeddings=dense_emb,
        multimask_output=False,
    )
    gt_256 = cv2.resize(gt_mask_hw.astype(np.float32), (256, 256), interpolation=cv2.INTER_NEAREST)
    gt_t = torch.from_numpy(gt_256).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    prob = torch.sigmoid(low_res_logits)
    bce = F.binary_cross_entropy(prob.clamp(1e-6, 1 - 1e-6), gt_t)
    inter = (prob * gt_t).sum()
    dice_loss = 1 - (2 * inter + 1e-5) / (prob.sum() + gt_t.sum() + 1e-5)
    return bce + dice_loss

@torch.no_grad()
def validate():
    medsam_model.mask_decoder.eval()
    names = sorted(set(n for n, _ in val_box_ds.all_samples))
    dices = []
    for img_name in names:
        embedding, H, W = get_embedding(img_name, VAL_IMG)
        data = get_json(img_name, VAL_JSON)
        pred = np.zeros((H, W), dtype=np.float32)
        gt = np.zeros((H, W), dtype=np.uint8)
        poly_idxs = [i for i, s in enumerate(data.get('shapes', [])) if s.get('shape_type') == 'polygon']
        for shape_idx in poly_idxs:
            points = np.array(data['shapes'][shape_idx]['points'])
            cv2.fillPoly(gt, [points.astype(np.int32)], 1)
            x_min, y_min = points.min(axis=0); x_max, y_max = points.max(axis=0)
            sample_idx = val_box_ds.all_samples.index((img_name, shape_idx))
            b = val_box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W, seed_idx=sample_idx)
            prob = medsam_predict_prob(embedding, [b[0], b[2], b[1], b[3]], H, W)
            pred = np.maximum(pred, prob)
        m = calc_metrics_img((pred > 0.5).astype(np.uint8), gt)
        dices.append(m['dice'])
    medsam_model.mask_decoder.train(True)
    return float(np.mean(dices))

print('fine-tuning helpers ready')

In [ ]:
# -- Part B: training loop with early stopping ------------------------------
best_val_dice = -1.0
epochs_without_improve = 0
train_samples = list(train_box_ds.all_samples)
steps_per_epoch = (len(train_samples) + BATCH_SIZE - 1) // BATCH_SIZE
n_train_images = len(set(n for n, _ in train_samples))
print(f'{steps_per_epoch} optimizer steps/epoch | epoch 1 also fills the embedding cache '
      f'(~{n_train_images} ViT-B forwards, several minutes); later epochs are decoder-only', flush=True)

for epoch in range(1, EPOCHS + 1):
    random.shuffle(train_samples)
    losses = []
    for step, batch_start in enumerate(range(0, len(train_samples), BATCH_SIZE)):
        batch = train_samples[batch_start:batch_start + BATCH_SIZE]
        batch_losses = []
        for img_name, shape_idx in batch:
            embedding, H, W = get_embedding(img_name, TRAIN_IMG)
            data = get_json(img_name, TRAIN_JSON)
            points = np.array(data['shapes'][shape_idx]['points'])
            gt = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(gt, [points.astype(np.int32)], 1)
            x_min, y_min = points.min(axis=0); x_max, y_max = points.max(axis=0)
            if random.random() < MIXED_SHIFT_PROB:
                b = train_box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W)
            else:
                b = train_box_ds._center_zoom_bbox(x_min, x_max, y_min, y_max, H, W)
            box_xyxy = [b[0], b[2], b[1], b[3]]
            batch_losses.append(medsam_compute_loss(embedding, box_xyxy, gt, H, W))

        optimizer.zero_grad()
        batch_loss = torch.stack(batch_losses).mean()
        batch_loss.backward()
        optimizer.step()
        losses.append(float(batch_loss.item()))

        if step < 3 or step % LOG_EVERY == 0:
            print(f'  epoch {epoch:3d} | step {step:4d}/{steps_per_epoch} | '
                  f'loss {np.mean(losses[-LOG_EVERY:]):.4f} | embeds cached {len(_embed_cache)}', flush=True)

    val_dice = validate()
    print(f'epoch {epoch:3d}/{EPOCHS} | train loss {np.mean(losses):.4f} | val Dice (off-center) {val_dice:.4f}', flush=True)
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        epochs_without_improve = 0
        torch.save(medsam_model.mask_decoder.state_dict(), BEST_CKPT_PATH)
        print(f'  -> new best, saved to {BEST_CKPT_PATH}')
    else:
        epochs_without_improve += 1
        if epochs_without_improve >= PATIENCE:
            print(f'early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)')
            break

print(f'\nBest validation Dice (off-center): {best_val_dice:.4f}')

## Part B - re-test the best fine-tuned checkpoint on the test split

Loads the best-validation-Dice mask-decoder weights and reruns the exact
same test procedure as Part A, so the two rows are directly comparable.

In [ ]:
# -- Part B: load best checkpoint, test --
medsam_model.mask_decoder.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=True))
medsam_model.mask_decoder.eval()
rows_finetuned, images_finetuned = run_test('fine-tuned')

# -- Qualitative figures (10 shared stems, same style as Finetune_SAMMed2D_test_robust.ipynb) --
visualize_qualitative(images_finetuned, TEST_IMG, box_ds, 'MedSAM (fine-tuned)',
                      f'medsam_finetuned_{DS_NAME.lower()}')

print('\n\n=== Zero-shot vs fine-tuned, side by side ===')
for rz, rf in zip(rows_zeroshot, rows_finetuned):
    assert rz['prompt'] == rf['prompt']
    print(f"{rz['prompt']:<12} zero-shot Dice {rz['dice']:.3f}  ->  fine-tuned Dice {rf['dice']:.3f}"
          f"  (delta {rf['dice']-rz['dice']:+.3f})")